# 02 — Qdrant: industry-wise schema + Word/PDF upload with LangChain

Everything is in this notebook: schema, embedding, LangChain loading, ingestion gates, Qdrant store, tests. Sections 0-4 are identical to `01_pgvector.ipynb` (same schema, same gates, same chunks), so results are comparable across databases.

| # | Section |
|---|---|
| 0 | Setup (uv `.venv` kernel) |
| 1 | Schema: base record + 8 industry models (Pydantic = single source of truth) |
| 2 | Embedding model (pinned, cached by content hash) |
| 3 | Load Word/PDF with **LangChain** + attach planned metadata from the manifest |
| 4 | Ingestion gates: source → purpose → classification → ACL → retention → sensitive scan → section chunking → validate |
| 5 | Qdrant store: one collection per industry, typed payload indexes (`tenant_id` with `is_tenant`), collection alias per industry, SQLite system of record |
| 6 | Upload all industry documents (bad docs → quarantine) |
| 7 | Idempotency · 8 Staging → promote · 9 Safe search · 10 Isolation · 11 Update · 12 Delete · 13 Metrics |

Inputs: `data/docs/<industry>/*.pdf|*.docx` + `manifest.jsonl` (made by `scripts/make_docs.py`, all content fake).
Server: Qdrant at `http://127.0.0.1:6333` (`QDRANT_URL` overrides). System of record: `data/sor/qdrant.db` (SQLite, redacted text only).
Design: [docs/PLAN.md](../docs/PLAN.md) · Setup: [SETUP.md](../SETUP.md)

## 0. Setup

In [ ]:
import os, re, json, math, time, hashlib, statistics, types, warnings
from datetime import date, datetime, timezone
from pathlib import Path
from typing import ClassVar, Literal, Union, get_args, get_origin
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")
os.environ.setdefault("HF_HUB_OFFLINE", "1")   # model already cached locally; delete this line on a new machine
DOCS, SAMPLES, RESULTS = ROOT / "data" / "docs", ROOT / "data" / "samples", ROOT / "data" / "results"
pd.set_option("display.max_colwidth", 90)

import sys; print("python:", sys.executable)

## 1. Schema — one record = one retrievable chunk

Base groups from the blueprint: **identity, provenance, content/vector, access/context, lifecycle**. Each industry adds its own typed fields and rules. Validation is the last gate: anything invalid never reaches the index.

In [ ]:
SCHEMA_VERSION = 1
TOKEN = r"^tok_[a-z0-9]+$"          # tokenized identifier, never a raw ID
Status = Literal["staged", "active", "superseded", "deleted"]
Classification = Literal["public", "internal", "confidential", "restricted"]


def make_chunk_id(tenant_id, source_id, source_version, chunk_no) -> str:
    """Deterministic -> idempotent upserts."""
    return hashlib.sha256(f"{tenant_id}|{source_id}|{source_version}|{chunk_no}".encode()).hexdigest()


def content_hash(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()


class BaseChunk(BaseModel):
    model_config = ConfigDict(extra="forbid")
    INDUSTRY: ClassVar[str] = ""
    # identity
    chunk_id: str
    tenant_id: str = Field(min_length=1)
    industry: str
    entity_type: str
    # provenance
    source_id: str
    source_version: str
    source_file: str
    chunk_no: int = Field(ge=0)
    section: str | None = None
    content_hash: str
    parent_id: str | None = None
    # content / vector
    text: str = Field(min_length=1)
    embedding: list[float]
    embedding_model: str
    language: str = "en"
    # access / context
    classification: Classification
    allowed_groups: list[str] = Field(min_length=1)        # fail closed: no ACL -> reject
    jurisdiction: str
    valid_from: date
    valid_to: date | None = None
    # lifecycle
    retention_until: date
    status: Status = "staged"
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

    @model_validator(mode="after")
    def _base_guardrails(self):
        if self.chunk_id != make_chunk_id(self.tenant_id, self.source_id, self.source_version, self.chunk_no):
            raise ValueError("chunk_id is not the stable hash of tenant|source|version|chunk_no")
        if self.content_hash != content_hash(self.text):
            raise ValueError("content_hash does not match text")
        if not self.embedding or not all(math.isfinite(x) for x in self.embedding):
            raise ValueError("embedding empty or non-finite")
        if self.valid_to and self.valid_to < self.valid_from:
            raise ValueError("valid_to before valid_from")
        if self.retention_until < date.today():
            raise ValueError("retention_until already passed")
        return self


BASE_FIELDS = frozenset(BaseChunk.model_fields)

In [ ]:
class HealthcareChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "healthcare"
    industry: Literal["healthcare"] = "healthcare"
    patient_ref_token: str = Field(pattern=TOKEN)
    record_type: str
    provider: str
    encounter_date: date
    consent_scope: Literal["treatment", "care_coordination", "research"]


class BankingChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "banking"
    industry: Literal["banking"] = "banking"
    customer_ref_token: str | None = Field(default=None, pattern=TOKEN)
    product: str
    account_type: str
    document_type: str
    effective_date: date


class InsuranceChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "insurance"
    industry: Literal["insurance"] = "insurance"
    policy_ref: str = Field(pattern=TOKEN)
    claim_ref: str | None = Field(default=None, pattern=TOKEN)
    line_of_business: str
    event_date: date | None = None
    claim_status: Literal["open", "closed", "n/a"]


class LegalChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "legal"
    industry: Literal["legal"] = "legal"
    matter_id: str
    client_ref: str = Field(pattern=TOKEN)
    privilege_status: Literal["none", "privileged", "work_product"]
    document_type: str
    effective_date: date

    @model_validator(mode="after")
    def _privilege(self):
        if self.privilege_status != "none" and self.classification != "restricted":
            raise ValueError("privileged/work-product content must be classified 'restricted'")
        return self


class RetailChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "retail"
    industry: Literal["retail"] = "retail"
    product_id: str
    sku: str
    brand: str
    locale: str
    channel: Literal["online", "store", "omni"]
    price_valid_from: date | None = None
    price_valid_to: date | None = None
    inventory_region: str


class ManufacturingChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "manufacturing"
    industry: Literal["manufacturing"] = "manufacturing"
    asset_id: str
    site_id: str
    equipment_type: str
    safety_class: Literal["low", "medium", "high"]
    revision: str
    document_owner: str
    approval_status: Literal["approved"]          # draft/obsolete procedures are never indexed


class GovernmentChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "government"
    industry: Literal["government"] = "government"
    agency_id: str
    record_class: str
    release_status: Literal["public_release", "restricted_release"]
    case_ref_token: str | None = Field(default=None, pattern=TOKEN)

    @model_validator(mode="after")
    def _release(self):
        if self.classification == "public" and self.release_status != "public_release":
            raise ValueError("public classification requires release_status=public_release")
        return self


class EducationHrChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "education_hr"
    industry: Literal["education_hr"] = "education_hr"
    domain: Literal["education", "hr"]
    org_unit: str
    course_or_department: str
    term: str | None = None
    region: str
    subject_ref_token: str | None = Field(default=None, pattern=TOKEN)


INDUSTRIES: dict[str, type[BaseChunk]] = {m.INDUSTRY: m for m in (
    HealthcareChunk, BankingChunk, InsuranceChunk, LegalChunk,
    RetailChunk, ManufacturingChunk, GovernmentChunk, EducationHrChunk)}

print("base fields:", ", ".join(BaseChunk.model_fields))
pd.DataFrame([{"industry": i, "model": m.__name__,
               "industry fields": ", ".join(f for f in m.model_fields if f not in BASE_FIELDS)}
              for i, m in INDUSTRIES.items()])

## 2. Embedding model — pinned, normalized, cached by content hash

In [ ]:
from sentence_transformers import SentenceTransformer


class Embedder:
    QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # bge v1.5 query instruction

    def __init__(self, model_name="BAAI/bge-small-en-v1.5", tag="bgesmall"):
        self.model_name, self.tag = model_name, tag
        self.model = SentenceTransformer(model_name)
        self.dim = (getattr(self.model, "get_embedding_dimension", None) or self.model.get_sentence_embedding_dimension)()
        self._cache, self.cache_hits = {}, 0

    def embed(self, texts):
        keys = [content_hash(t) for t in texts]
        todo = [(k, t) for k, t in zip(keys, texts) if k not in self._cache]
        self.cache_hits += len(texts) - len(todo)
        if todo:
            for (k, _), v in zip(todo, self.model.encode([t for _, t in todo], normalize_embeddings=True)):
                self._cache[k] = v.astype(float).tolist()
        return [self._cache[k] for k in keys]

    def embed_query(self, q):
        return self.model.encode(self.QUERY_PREFIX + q, normalize_embeddings=True).astype(float).tolist()


embedder = Embedder()
print(embedder.model_name, "| dim", embedder.dim)

## 3. Load Word / PDF with LangChain + attach the planned metadata

- `PyPDFLoader` (one `Document` per page) and `Docx2txtLoader` (one `Document` per file).
- `manifest.jsonl` is the source system's metadata export: tenant, ACL, classification, jurisdiction, validity, retention, industry fields. It is merged into `Document.metadata`.
- A file **without** a manifest entry is rejected: no metadata = no ACL = fail closed.
- **Clean** step: drop repeated header/footer lines (banner, address/footer bars, signature line) so they never get embedded.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)          # langchain-community sunset notice
    from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

LOADERS = {".pdf": PyPDFLoader, ".docx": Docx2txtLoader}
CLASS_WORDS = {"PUBLIC", "INTERNAL", "CONFIDENTIAL", "RESTRICTED"}


@dataclass
class Rejection:
    industry: str
    source_id: str
    gate: str
    reason: str


def clean_text(text: str) -> str:
    """Blueprint 'CLEAN' step: remove page furniture, keep content."""
    keep = []
    for line in text.splitlines():
        s = line.strip()
        if not s or s in CLASS_WORDS:                       # empty / classification banner
            continue
        if "  |  " in s or s.startswith("Approved by:"):    # address bar, footer bar, signature line
            continue
        keep.append(re.sub(r"\s{2,}", " ", s))
    return "\n".join(keep)


def load_langchain_documents(industry: str):
    folder = DOCS / industry
    manifest = {m["file"]: m for m in (json.loads(l) for l in (folder / "manifest.jsonl").read_text("utf-8").splitlines() if l)}
    docs, rejections = [], []
    for path in sorted(p for p in folder.iterdir() if p.suffix.lower() in LOADERS):
        meta = manifest.get(path.name)
        if meta is None:
            rejections.append(Rejection(industry, path.stem, "metadata", f"{path.name} has no manifest entry")); continue
        loader = LOADERS[path.suffix.lower()]
        for page in loader(str(path)).load():
            page.metadata = {**page.metadata, **meta, "loader": loader.__name__}
            page.page_content = clean_text(page.page_content)
            docs.append(page)
    return docs, rejections


def to_source_docs(lc_docs: list[Document]) -> list[dict]:
    """Merge pages of the same file into one source doc (pipeline input)."""
    merged = {}
    for d in lc_docs:
        m = merged.setdefault(d.metadata["file"], {**{k: v for k, v in d.metadata.items()
                                                       if k not in ("page", "page_label", "total_pages")}, "text": ""})
        m["text"] = (m["text"] + "\n" + d.page_content).strip()
    return list(merged.values())


lc_docs, _ = load_langchain_documents("healthcare")
d0 = lc_docs[0]
print("page_content (cleaned):\n", d0.page_content[:600], "...\n")
{k: d0.metadata[k] for k in ["source", "loader", "file", "source_id", "tenant_id", "classification",
                             "allowed_groups", "jurisdiction", "valid_from", "retention_until", "metadata"]}

## 4. Ingestion gates (fail closed)

Order: **authorized source → approved purpose → classified → ACL present → retention valid → sensitive scan (reject secrets/SSN/cards, redact email/phone) → section-aware chunking (LangChain splitter) → embed → schema validation**. Each chunk text is prefixed with `title - section` so it stays meaningful on its own.

In [ ]:
APPROVED_SOURCES = {"ehr_export", "core_banking", "claims_system", "dms_legal", "pim", "plm", "records_mgmt", "lms_hris"}
APPROVED_PURPOSES = {"rag_search", "support_assistant"}
SECRETS = [re.compile(p) for p in (r"AKIA[0-9A-Z]{16}", r"-----BEGIN [A-Z ]*PRIVATE KEY-----",
                                   r"(?i)\b(password|passwd|api[_-]?key|secret)\s*[:=]\s*\S+")]
SSN = re.compile(r"\b\d{3}-\d{2}-\d{4}\b")
CARD = re.compile(r"\b(?:\d[ -]?){13,19}\b")
EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.]+\b")
PHONE = re.compile(r"\+?\d{1,2}[ .-]?\(?\d{3}\)?[ .-]?\d{3}[ .-]?\d{4}\b")
SECTION = re.compile(r"^(\d+\. [A-Z][A-Za-z ]+|Revision History)$", re.M)

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40,
                                          separators=[". ", "; ", " ", ""], keep_separator="end")


def luhn(digits: str) -> bool:
    nums = [int(x) for x in digits][::-1]
    return sum(n if i % 2 == 0 else (n * 2 - 9 if n * 2 > 9 else n * 2) for i, n in enumerate(nums)) % 10 == 0


def sensitive_scan(text):
    """(reject_reason | None, redacted_text)"""
    if any(p.search(text) for p in SECRETS):
        return "credential/secret detected", text
    if SSN.search(text):
        return "SSN detected", text
    for m in CARD.finditer(text):
        digits = re.sub(r"\D", "", m.group())
        if 13 <= len(digits) <= 19 and luhn(digits):
            return "payment card number detected", text
    return None, PHONE.sub("[PHONE]", EMAIL.sub("[EMAIL]", text))


def split_sections(text, title):
    """[(section, body)] using the document's numbered headings; text before the first heading = control block."""
    marks = list(SECTION.finditer(text))
    parts = [("Document control", text[:marks[0].start()] if marks else text)]
    for i, m in enumerate(marks):
        parts.append((m.group(1), text[m.end(): marks[i + 1].start() if i + 1 < len(marks) else len(text)]))
    return [(s, " ".join(b.replace(title, "", 1).split() if s == "Document control" else b.split()))
            for s, b in parts if b.strip()]


@dataclass
class IngestResult:
    records: list = field(default_factory=list)
    rejections: list = field(default_factory=list)
    clean_docs: list = field(default_factory=list)


def run_pipeline(docs, industry, embedder) -> IngestResult:
    model, out = INDUSTRIES[industry], IngestResult()
    for doc in docs:
        sid = doc.get("source_id", "?")
        reject = lambda gate, why: out.rejections.append(Rejection(industry, sid, gate, why))
        if doc.get("source_system") not in APPROVED_SOURCES:
            reject("authorized_source", f"source_system '{doc.get('source_system')}' not approved"); continue
        if doc.get("purpose") not in APPROVED_PURPOSES:
            reject("approved_purpose", f"purpose '{doc.get('purpose')}' not approved"); continue
        if not doc.get("classification"):
            reject("classified", "missing classification"); continue
        if not doc.get("allowed_groups"):
            reject("acl", "no allowed_groups (ACL)"); continue
        if date.fromisoformat(doc["retention_until"]) < date.today():
            reject("retention", f"retention_until {doc['retention_until']} already passed"); continue
        why, text = sensitive_scan(doc["text"])
        if why:
            reject("sensitive_scan", why); continue

        pieces = [(sec, f"{doc['title']} - {sec}: {p}") for sec, body in split_sections(text, doc["title"])
                  for p in splitter.split_text(body)]
        vecs = embedder.embed([p for _, p in pieces])
        try:
            built = [model(
                chunk_id=make_chunk_id(doc["tenant_id"], sid, doc["source_version"], i),
                tenant_id=doc["tenant_id"], entity_type=doc["entity_type"], source_id=sid,
                source_version=doc["source_version"], source_file=doc["file"], chunk_no=i, section=sec,
                content_hash=content_hash(p), parent_id=sid, text=p, embedding=v,
                embedding_model=embedder.model_name, language=doc.get("language", "en"),
                classification=doc["classification"], allowed_groups=doc["allowed_groups"],
                jurisdiction=doc["jurisdiction"], valid_from=doc["valid_from"], valid_to=doc.get("valid_to"),
                retention_until=doc["retention_until"], **doc.get("metadata", {}))
                for i, ((sec, p), v) in enumerate(zip(pieces, vecs))]
        except ValidationError as e:
            err = e.errors()[0]
            reject("schema_validation", f"{'.'.join(map(str, err['loc'])) or 'record'}: {err['msg']}"); continue
        out.records += built
        out.clean_docs.append({**doc, "text": text})
    return out


demo = run_pipeline(to_source_docs(lc_docs), "healthcare", embedder)
pd.DataFrame([{"source_id": r.source_id, "chunk_no": r.chunk_no, "section": r.section, "text": r.text}
              for r in demo.records if r.source_id == "hc-001"])

## 5. Qdrant store — one collection per industry

Per industry: versioned collection `{industry}_v1_bgesmall_384` (cosine, 384-d) and a **collection alias** `{industry}` (the blue/green switch point; every read and write below goes through the alias).
Payload is the Pydantic record minus the vector, with typed payload indexes: `tenant_id` keyword **`is_tenant=True`** (Qdrant co-locates each tenant's points), `allowed_groups` / `status` / `source_id` / `source_version` keyword, and every `date` field of the model as a **datetime** index (stored as RFC 3339, so `valid_from <= today` is a range filter inside Qdrant).
Point id = `uuid5(chunk_id)` → idempotent upserts. `create_indexes` drops and recreates the collections, so the notebook can be re-run from scratch.
System of record = `data/sor/qdrant.db` (SQLite, tables `sources` + `quarantine`), separate from 01's Postgres tables.

In [ ]:
import sqlite3, uuid
from qdrant_client import QdrantClient, models as qm

QDRANT_URL = os.environ.get("QDRANT_URL", "http://127.0.0.1:6333")   # not 'localhost': avoids a 2 s IPv6 fallback per request on Windows
SOR_PATH = ROOT / "data" / "sor" / "qdrant.db"
POINT_NS = uuid.UUID("6f1c8a52-0d4e-4b8e-9a4f-2b7f8f0c5e11")      # fixed namespace -> same chunk_id, same point id


def index_name(industry):
    return f"{industry}_v{SCHEMA_VERSION}_{embedder.tag}_{embedder.dim}"


def point_id(chunk_id: str) -> str:
    return str(uuid.uuid5(POINT_NS, chunk_id))


def _is_date(ann):
    if get_origin(ann) in (Union, types.UnionType):
        ann = [a for a in get_args(ann) if a is not type(None)][0]
    return ann is date


def date_fields(model):
    return [n for n, f in model.model_fields.items() if _is_date(f.annotation)]


def rfc3339(d: date | str | None):
    return None if d is None else f"{d if isinstance(d, str) else d.isoformat()}T00:00:00Z"


KEYWORD_FIELDS = ["allowed_groups", "status", "source_id", "source_version", "classification"]


def connect():
    client = QdrantClient(url=QDRANT_URL, timeout=60)
    SOR_PATH.parent.mkdir(parents=True, exist_ok=True)
    sor = sqlite3.connect(SOR_PATH)
    return client, sor


SOR_DDL = """
DROP TABLE IF EXISTS sources;
DROP TABLE IF EXISTS quarantine;
CREATE TABLE sources (
  source_id TEXT NOT NULL, source_version TEXT NOT NULL, tenant_id TEXT NOT NULL, industry TEXT NOT NULL,
  title TEXT, source_file TEXT, text TEXT NOT NULL, raw TEXT NOT NULL, retention_until TEXT NOT NULL,
  PRIMARY KEY (source_id, source_version));
CREATE TABLE quarantine (
  industry TEXT, source_id TEXT, gate TEXT, reason TEXT, UNIQUE (industry, source_id, gate, reason));
"""


def create_indexes(client, sor):
    """Recreate cleanly: collection per industry + payload indexes + alias {industry} -> collection."""
    sor.executescript(SOR_DDL); sor.commit()
    for ind, model in INDUSTRIES.items():
        coll = index_name(ind)
        if client.collection_exists(coll):
            client.delete_collection(coll)
        client.create_collection(coll, vectors_config=qm.VectorParams(size=embedder.dim, distance=qm.Distance.COSINE))
        client.create_payload_index(coll, "tenant_id", field_schema=qm.KeywordIndexParams(type="keyword", is_tenant=True))
        for f in KEYWORD_FIELDS:
            client.create_payload_index(coll, f, field_schema=qm.PayloadSchemaType.KEYWORD)
        for f in date_fields(model):
            client.create_payload_index(coll, f, field_schema=qm.PayloadSchemaType.DATETIME)
        client.update_collection_aliases(change_aliases_operations=[
            qm.DeleteAliasOperation(delete_alias=qm.DeleteAlias(alias_name=ind)),
            qm.CreateAliasOperation(create_alias=qm.CreateAlias(collection_name=coll, alias_name=ind))])
    return {i: index_name(i) for i in INDUSTRIES}


def save_sources(sor, industry, docs):
    sor.executemany(
        "INSERT INTO sources (source_id, source_version, tenant_id, industry, title, source_file, text, raw, retention_until) "
        "VALUES (?,?,?,?,?,?,?,?,?) ON CONFLICT (source_id, source_version) DO UPDATE "
        "SET text = excluded.text, raw = excluded.raw, retention_until = excluded.retention_until",
        [(d["source_id"], d["source_version"], d["tenant_id"], industry, d.get("title"), d.get("file"), d["text"],
          json.dumps({k: v for k, v in d.items() if k != "text"}, default=str), d["retention_until"]) for d in docs])
    sor.commit()


def save_rejections(sor, rejections):
    sor.executemany("INSERT OR IGNORE INTO quarantine (industry, source_id, gate, reason) VALUES (?,?,?,?)",
                    [(r.industry, r.source_id, r.gate, r.reason) for r in rejections])
    sor.commit()


def to_payload(r: BaseChunk) -> dict:
    p = r.model_dump(mode="json", exclude={"embedding"})
    for f in date_fields(type(r)):
        p[f] = rfc3339(p[f])
    return p


def upsert(client, industry, records):
    """Idempotent: point id = uuid5(chunk_id). Existing status/created_at are kept (like ON CONFLICT ... DO UPDATE)."""
    if not records:
        return 0
    ids = [point_id(r.chunk_id) for r in records]
    keep = {p.id: p.payload for p in client.retrieve(industry, ids=ids, with_payload=["status", "created_at"])}
    points = []
    for pid, r in zip(ids, records):
        payload = to_payload(r) | keep.get(pid, {})
        points.append(qm.PointStruct(id=pid, vector=r.embedding, payload=payload))
    client.upsert(industry, points=points, wait=True)
    return len(points)


def _match(key, value):
    return qm.FieldCondition(key=key, match=qm.MatchValue(value=value))


def _n(client, industry, flt):
    return client.count(industry, count_filter=flt, exact=True).count


def promote(client, industry, source_ids=None):
    must = [_match("status", "staged")]
    if source_ids:
        must.append(qm.FieldCondition(key="source_id", match=qm.MatchAny(any=list(source_ids))))
    flt = qm.Filter(must=must)
    n = _n(client, industry, flt)
    client.set_payload(industry, payload={"status": "active"}, points=flt, wait=True)
    return n


def supersede(client, industry, source_id, keep_version):
    flt = qm.Filter(must=[_match("source_id", source_id), _match("status", "active")],
                    must_not=[_match("source_version", keep_version)])
    n = _n(client, industry, flt)
    client.set_payload(industry, payload={"status": "superseded"}, points=flt, wait=True)
    return n


def purge(client, sor, industry, source_id):
    flt = qm.Filter(must=[_match("source_id", source_id)])
    n = _n(client, industry, flt)
    client.delete(industry, points_selector=qm.FilterSelector(filter=flt), wait=True)
    sor.execute("DELETE FROM sources WHERE source_id = ?", (source_id,)); sor.commit()
    return n


def count(client, industry, tenant_id=None):
    return _n(client, industry, qm.Filter(must=[_match("tenant_id", tenant_id)]) if tenant_id else None)


def search_filter(tenant_id, groups, today=None):
    now = datetime.combine(today or date.today(), datetime.min.time(), tzinfo=timezone.utc)
    return qm.Filter(must=[
        _match("tenant_id", tenant_id),
        qm.FieldCondition(key="allowed_groups", match=qm.MatchAny(any=list(groups))),     # any-match ACL
        _match("status", "active"),
        qm.FieldCondition(key="valid_from", range=qm.DatetimeRange(lte=now)),
        qm.Filter(should=[qm.IsNullCondition(is_null=qm.PayloadField(key="valid_to")),
                          qm.IsEmptyCondition(is_empty=qm.PayloadField(key="valid_to")),
                          qm.FieldCondition(key="valid_to", range=qm.DatetimeRange(gte=now))]),
    ])


def safe_search(client, industry, tenant_id, groups, query_vec, k=5):
    """The ONLY query path. Tenant + ACL + status + validity are a Qdrant pre-filter applied during the HNSW
    search (filterable HNSW), never a Python post-filter after top-k."""
    if not tenant_id or not groups:
        raise PermissionError("safe_search needs tenant_id and at least one group")
    res = client.query_points(industry, query=list(query_vec), query_filter=search_filter(tenant_id, groups),
                              limit=k, with_payload=["chunk_id", "tenant_id", "source_id", "source_version", "section", "text"])
    return [{**{f: p.payload.get(f) for f in ("chunk_id", "tenant_id", "source_id", "source_version", "section", "text")},
             "score": p.score} for p in res.points]


client, sor = connect()
info = client.info()
print("qdrant server", info.version, "| qdrant-client", __import__("importlib.metadata").metadata.version("qdrant-client"))
print("payload for HealthcareChunk:", [f for f in HealthcareChunk.model_fields if f != "embedding"])
print("datetime-indexed fields:", date_fields(HealthcareChunk))

In [ ]:
collections = create_indexes(client, sor)
aliases = {a.alias_name: a.collection_name for a in client.get_aliases().aliases}
pd.DataFrame([{"industry (alias)": i, "collection": c, "alias -> collection": aliases.get(i),
               "payload indexes": ", ".join(f"{k}:{v.data_type.value}{'(tenant)' if getattr(v.params, 'is_tenant', False) else ''}"
                                            for k, v in sorted(client.get_collection(c).payload_schema.items()))}
              for i, c in collections.items()])

## 6. Upload all industry documents (Word/PDF → LangChain → gates → Qdrant)

In [ ]:
results, summary = {}, []
for ind in INDUSTRIES:
    lc, missing_meta = load_langchain_documents(ind)
    docs = to_source_docs(lc)
    res = run_pipeline(docs, ind, embedder)
    res.rejections += missing_meta
    save_sources(sor, ind, res.clean_docs)
    upsert(client, ind, res.records)
    save_rejections(sor, res.rejections)
    results[ind] = res
    summary.append({"industry": ind, "files": len(docs),
                    "pdf": sum(d["file"].endswith(".pdf") for d in docs), "docx": sum(d["file"].endswith(".docx") for d in docs),
                    "accepted": len(res.clean_docs), "chunks": len(res.records), "quarantined": len(res.rejections),
                    "tenants": ", ".join(sorted({d["tenant_id"] for d in docs}))})
summary_df = pd.DataFrame(summary)
assert (summary_df["quarantined"] >= 1).all() and all(count(client, i) == len(results[i].records) for i in INDUSTRIES)
summary_df

In [ ]:
# quarantine (fail closed): every industry's bad document stopped at a gate (SQLite system of record)
pd.DataFrame(sor.execute("select industry, source_id, gate, reason from quarantine order by industry").fetchall(),
             columns=["industry", "source_id", "gate", "reason"])

In [ ]:
# stored points carry the industry-specific fields as typed payload
pts, _ = client.scroll("healthcare", limit=200, with_payload=True)
pd.DataFrame([{k: p.payload[k] for k in ["source_id", "source_file", "chunk_no", "section", "tenant_id", "record_type",
                                          "patient_ref_token", "consent_scope", "encounter_date", "allowed_groups", "status"]}
              for p in pts]).sort_values(["source_id", "chunk_no"]).reset_index(drop=True)

In [ ]:
# redaction: the email inside hc-001.pdf never reaches the index or the system of record
txt = sor.execute("select text from sources where source_id = 'hc-001'").fetchone()[0]
assert "@" not in txt and "[EMAIL]" in txt
pts, _ = client.scroll("healthcare", scroll_filter=qm.Filter(must=[_match("source_id", "hc-001")]), limit=100, with_payload=["text"])
assert pts and all("@" not in p.payload["text"] for p in pts)
print(re.search(r"[^.]*\[EMAIL\][^.]*\.?", txt).group().strip())

## 7. Idempotency — same upload again, same point counts

In [ ]:
before = {i: count(client, i) for i in INDUSTRIES}
for ind, res in results.items():
    upsert(client, ind, res.records)
after = {i: count(client, i) for i in INDUSTRIES}
assert before == after
print("idempotent ✔", after, "| embedding cache hits:", embedder.cache_hits)

## 8. Staging → promote — staged chunks are never searchable

In [ ]:
golden = json.loads((SAMPLES / "golden_queries.json").read_text())
staged_hits = {g["query"]: len(safe_search(client, g["industry"], g["tenant_id"], g["groups"], embedder.embed_query(g["query"])))
               for g in golden}
assert sum(staged_hits.values()) == 0, staged_hits
print("hits while staged (all 16 golden queries):", sum(staged_hits.values()))
print("promoted:", {i: promote(client, i) for i in INDUSTRIES})
pd.DataFrame([{"industry": i, "status": s, "chunks": n} for i in INDUSTRIES for s in ("staged", "active")
              if (n := _n(client, i, qm.Filter(must=[_match("status", s)])))])

## 9. Safe search — golden queries per industry

In [ ]:
rows, latencies = [], []
for g in golden:
    qv = embedder.embed_query(g["query"])
    t0 = time.perf_counter()
    hits = safe_search(client, g["industry"], g["tenant_id"], g["groups"], qv, k=5)
    latencies.append((time.perf_counter() - t0) * 1000)
    found = [h["source_id"] for h in hits]
    rows.append({"industry": g["industry"], "tenant": g["tenant_id"], "query": g["query"], "expected": g["expected_source_id"],
                 "top1": found[0] if found else None, "top1 section": hits[0]["section"] if hits else None,
                 "score": round(hits[0]["score"], 3) if hits else None,
                 "hit@1": found[:1] == [g["expected_source_id"]], "hit@5": g["expected_source_id"] in found})
search_df = pd.DataFrame(rows)
assert search_df["hit@5"].all(), search_df[~search_df["hit@5"]]
print(f"hit@1 = {search_df['hit@1'].mean():.2f} | hit@5 = {search_df['hit@5'].mean():.2f} | p50 latency = {statistics.median(latencies):.1f} ms")
search_df

In [ ]:
pd.DataFrame(safe_search(client, "manufacturing", "forgeworks", ["maintenance"], embedder.embed_query("how to lock out the hydraulic press")))

## 10. Isolation tests

**What enforces isolation in Qdrant (honest statement):** Qdrant has no server-side row-level security. Isolation is (a) the **mandatory tenant + ACL filter inside `safe_search`**, the only query path the application exposes, applied as a pre-filter during the HNSW search, plus (b) the `tenant_id` payload index with **`is_tenant=True`**, which makes Qdrant co-locate each tenant's points and build per-tenant search structures. Anyone holding the API key can still read every tenant's points through the raw API, so production needs per-tenant JWT/RBAC keys or a gateway in front.

1. **Tenant-scoped counts**: `count` with a tenant filter returns only that tenant's points; the tenants add up to the collection total (no orphan or untagged points).
2. **Cross-tenant**: other tenant asks the same golden question → never gets the owner's chunks.
3. **ACL**: user without the group gets nothing. 4. **Fail closed**: no tenant/groups → `PermissionError`.

In [ ]:
tc = []
for ind in INDUSTRIES:
    tenants = sorted({d["tenant_id"] for d in results[ind].clean_docs})
    row = {"industry": ind, "total (unfiltered API)": count(client, ind)}
    row |= {t: count(client, ind, t) for t in tenants}
    row["unknown tenant"] = count(client, ind, "tenant_that_does_not_exist")
    assert sum(row[t] for t in tenants) == row["total (unfiltered API)"] and row["unknown tenant"] == 0
    for t in tenants:
        assert row[t] == sum(r.tenant_id == t for r in results[ind].records)    # tenant filter returns exactly its own points
    tc.append(row)
tenant_counts_df = pd.DataFrame(tc).fillna("-")
tenant_counts_df

In [ ]:
iso = []
for g in golden:
    other = ({d["tenant_id"] for d in results[g["industry"]].clean_docs} - {g["tenant_id"]}).pop()
    qv = embedder.embed_query(g["query"])
    hits = safe_search(client, g["industry"], other, g["groups"], qv)
    leaked = [h for h in hits if h["tenant_id"] != other or h["source_id"] == g["expected_source_id"]]
    no_acl = safe_search(client, g["industry"], g["tenant_id"], ["group_nobody_has"], qv)
    iso.append({"industry": g["industry"], "owner": g["tenant_id"], "asked as": other,
                "hits for other tenant": len(hits), "leaked": len(leaked), "hits without ACL group": len(no_acl)})
iso_df = pd.DataFrame(iso)
assert iso_df["leaked"].eq(0).all() and iso_df["hits without ACL group"].eq(0).all()
for bad in [("", ["care_team_cardiology"]), ("acme_health", [])]:
    try:
        safe_search(client, "healthcare", *bad, embedder.embed_query("x")); raise AssertionError("should raise")
    except PermissionError as e:
        print("fail closed ✔", e)
iso_df

## 11. Update — new version of `hc-002.docx` supersedes v1

In [ ]:
v1 = next(d for d in to_source_docs(load_langchain_documents("healthcare")[0]) if d["source_id"] == "hc-002")
v2 = {**v1, "source_version": "2",
      "text": v1["text"].replace("Metformin dose increased to 1000 mg twice daily", "Metformin dose increased to 1500 mg extended release once daily")}
res2 = run_pipeline([v2], "healthcare", embedder)
save_sources(sor, "healthcare", res2.clean_docs)
upsert(client, "healthcare", res2.records)
promote(client, "healthcare", ["hc-002"])
print("superseded chunks:", supersede(client, "healthcare", "hc-002", keep_version="2"))
hits = safe_search(client, "healthcare", "acme_health", ["care_team_endocrine"], embedder.embed_query("metformin dose"))
assert hits and any(h["source_id"] == "hc-002" for h in hits)
assert all(h["source_version"] == "2" for h in hits if h["source_id"] == "hc-002")
display(pd.DataFrame([{"version": v, "status": s, "chunks": n} for v in ("1", "2") for s in ("active", "superseded")
                      if (n := _n(client, "healthcare", qm.Filter(must=[_match("source_id", "hc-002"),
                                                                        _match("source_version", v), _match("status", s)])))]))
pd.DataFrame(hits)[["source_id", "source_version", "section", "score", "text"]]

## 12. Delete / consent withdrawal — purge `hc-004` and verify absence

In [ ]:
q = embedder.embed_query("ejection fraction systolic dysfunction")
before = [h["source_id"] for h in safe_search(client, "healthcare", "beacon_clinic", ["care_team_cardiology"], q)]
purged = purge(client, sor, "healthcare", "hc-004")
after = [h["source_id"] for h in safe_search(client, "healthcare", "beacon_clinic", ["care_team_cardiology"], q)]
left_points = _n(client, "healthcare", qm.Filter(must=[_match("source_id", "hc-004")]))
left = sor.execute("select count(*) from sources where source_id = 'hc-004'").fetchone()[0]
assert purged > 0 and "hc-004" in before and "hc-004" not in after and left_points == 0 and left == 0
print(f"purged {purged} points | before: {sorted(set(before))} | after: {sorted(set(after))} | "
      f"points left: {left_points} | SoR rows left: {left} ✔")

## 13. Metrics (for the cross-DB comparison notebook)

In [ ]:
metrics = {"db": "qdrant", "industries": len(INDUSTRIES), "chunks_total": sum(count(client, i) for i in INDUSTRIES),
           "quarantined": sum(len(r.rejections) for r in results.values()),
           "hit@1": float(search_df["hit@1"].mean()), "hit@5": float(search_df["hit@5"].mean()),
           "latency_p50_ms": round(statistics.median(latencies), 2),
           "isolation": "PASS (mandatory tenant+ACL payload pre-filter in safe_search + is_tenant keyword index; no server-side RLS)",
           "executed": True}
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "qdrant.json").write_text(json.dumps(metrics, indent=2))
metrics

## Next: embedding model change (blue/green)
Never re-embed into the existing collections. Build `{industry}_v1_<newmodel>_<dim>` with a new `Embedder`, re-embed from `data/sor/qdrant.db` (system of record), run the golden queries against the new collection, then switch the alias atomically in one `update_collection_aliases` call (`DeleteAlias {industry}` + `CreateAlias {industry} -> new collection`) and drop the old collection.